# XN on Google Colab

Train a small GPT-style model from scratch with [XN](https://github.com/nk4dev/xn) on a free Colab GPU.

1. **Runtime → Change runtime type → T4 GPU** (CPU also works, just slower).
2. Run the cells from top to bottom.

Colab already ships PyTorch with CUDA, so the Python engine is used directly (`python -m xn ...`) instead of the native `xn` CLI.

In [ ]:
!nvidia-smi -L || echo 'no GPU - training will run on the CPU'

## 1. Install XN

In [ ]:
!git clone -q --depth 1 https://github.com/nk4dev/xn.git /content/xn-src
!pip install -q -e /content/xn-src
!python -m xn doctor

## 2. Keep checkpoints on Google Drive (optional, recommended)

Colab sessions end without warning. With the project on Drive, `last.pt` survives and `--resume` (step 5) picks up where training stopped.

In [ ]:
USE_DRIVE = True  # set to False to keep everything in the temporary /content
PRESET = "xn-llm-code"  # "xn-llm-code" (JS/TS code) or "xn-llm" (Japanese text)

import os, shutil
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = f"/content/drive/MyDrive/xn/{PRESET}"
else:
    PROJECT = f"/content/{PRESET}"
os.makedirs(PROJECT, exist_ok=True)
if not os.path.exists(f"{PROJECT}/xn.json"):
    shutil.copy(f"/content/xn-src/templates/presets/{PRESET}.json", f"{PROJECT}/xn.json")
%cd $PROJECT
print(open("xn.json").read())

## 3. Get training data

- **xn-llm-code**: `xn data repo` downloads JS/TS source from GitHub repos with a permissive license (MIT, Apache-2.0, BSD, ...) and records the repo, commit and license in `data/SOURCES.json`. Copyleft or unlicensed repos are refused.
- **xn-llm**: `xn data add` downloads public-domain Japanese novels from Aozora Bunko.

In [ ]:
if PRESET == "xn-llm-code":
    for repo in ["expressjs/express", "axios/axios", "preactjs/preact", "immerjs/immer",
                 "reduxjs/redux", "pmndrs/zustand", "chalk/chalk"]:
        !python -m xn data repo {repo}
else:
    for url in ["https://www.aozora.gr.jp/cards/000148/files/752_ruby_2438.zip",   # 坊っちゃん
                "https://www.aozora.gr.jp/cards/000148/files/773_ruby_5968.zip",   # こころ
                "https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip",   # 吾輩は猫である
                "https://www.aozora.gr.jp/cards/000081/files/456_ruby_145.zip"]:   # 銀河鉄道の夜
        !python -m xn data add {url}
!python -m xn data list | tail -3

## 4. Train

Progress is printed every `eval_interval` steps. `ckpt.pt` keeps the best weights; `last.pt` (model + optimizer + step) is saved at every evaluation and when you press **■ Interrupt**.

Shorten a first try with e.g. `--max_steps 1000`.

In [ ]:
!python -m xn train

## 5. Resume after an interruption or a new session

Re-run cells 1 and 2 (same `PRESET`), then:

In [ ]:
!python -m xn train --resume

## 6. Generate

In [ ]:
prompt = "function " if PRESET == "xn-llm-code" else "吾輩は"
!python -m xn generate --prompt "{prompt}" --max_new_tokens 200 --temperature 0.8